In [1]:
import psycopg2
from psycopg2 import sql
from psycopg2.extras import execute_values, Json
import logging
import json
import html
import re

In [2]:
import pandas as pd
from pathlib import Path
import re

In [3]:
DB_PARAMS = {
    'dbname':   'thecall',
    'user':     'postgres',
    'password': 'password',
    'host':     'localhost',
    'port':     '5432'
}

TABLE_SCHEMA = 'articles'
TABLE_NAME   = 'proquestarticles_nite'
sourcedata = Path(r"D:\development\2026_python\2026_thecall\code\metadataanalysis\combined_metadata.csv")

conn = psycopg2.connect(**DB_PARAMS)
cur = conn.cursor()

In [4]:
# Cell: Strip YAML frontmatter from full_transcription column, updating null titles first

def extract_frontmatter_title(text):
    """Extract the TITLE value from YAML frontmatter."""
    if text is None:
        return None
    match = re.search(r'^TITLE:\s*(.+)$', text, flags=re.MULTILINE)
    return match.group(1).strip() if match else None

def strip_frontmatter(text):
    """Remove YAML frontmatter block (---...---) from the start of a markdown string."""
    if text is None:
        return None
    pattern = r'^\s*---\s*\n.*?\n---\s*\n'
    return re.sub(pattern, '', text, count=1, flags=re.DOTALL).lstrip('\n')

# Fetch all rows that have frontmatter
query = f"""
    SELECT id_proquest, full_transcript, title
    FROM {TABLE_SCHEMA}.{TABLE_NAME}
    WHERE full_transcript LIKE '---%'
"""
cur.execute(query)
rows = cur.fetchall()
print(f"Found {len(rows)} rows with frontmatter to process")

stripped = 0
titles_updated = 0

for row_id, text, existing_title in rows:
    
    # If title is null, extract it from frontmatter and update
    if existing_title is None:
        extracted_title = extract_frontmatter_title(text)
        if extracted_title:
            cur.execute(
                f"UPDATE {TABLE_SCHEMA}.{TABLE_NAME} SET title = %s WHERE id_proquest = %s",
                (extracted_title, row_id)
            )
            titles_updated += 1

    # Strip the frontmatter
    cleaned = strip_frontmatter(text)
    cur.execute(
        f"UPDATE {TABLE_SCHEMA}.{TABLE_NAME} SET full_transcript = %s WHERE id_proquest = %s",
        (cleaned, row_id)
    )
    stripped += 1

conn.commit()
print(f"Stripped frontmatter from {stripped} rows")
print(f"Updated null titles from frontmatter: {titles_updated} rows")

Found 336 rows with frontmatter to process
Stripped frontmatter from 336 rows
Updated null titles from frontmatter: 7 rows
